# Experiment 3: Signal Verification via XGBoost Tabular Regression

**Goal:** Definitively prove the dataset holds a valid, extractable regression signal *without* any transformer/LLM.

We transform raw source-code text into **TF-IDF** character n-gram features, augment them with hand-crafted **structural code metrics**, and train an **XGBoost** regressor to predict memory usage.

A moderate Spearman ρ (0.30–0.60) would validate that the signal exists and is learnable by a classical tabular model, independently confirming the dataset's utility for the neural RLM architecture.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.model_selection import train_test_split
import xgboost as xgb

from utils.data_loader import load_partition, print_mode_banner, is_validation_mode
from utils.features import compute_tfidf_features, compute_code_metrics_batch
from utils.plotting import (setup_style, plot_actual_vs_predicted, 
                            plot_feature_importance, COLORS)
import matplotlib.pyplot as plt

VALIDATION_MODE = is_validation_mode()
print_mode_banner(VALIDATION_MODE)
setup_style()

## 1. Load CDSS Data (Stratified Sample)

In [ ]:
if VALIDATION_MODE:
    sample_size = 500
else:
    sample_size = 250000

# In validation mode, load_partition already samples ~500 rows
# In full mode, load all then sample down to 250K
df = load_partition("CDSS", validation_mode=VALIDATION_MODE)
print(f"CDSS partition loaded: {len(df):,} rows")

# Further sampling if needed (only relevant in full mode)
if not VALIDATION_MODE and len(df) > sample_size:
    df = df.sample(n=sample_size, random_state=42).reset_index(drop=True)
    print(f"Sampled down to: {len(df):,} rows")

# Drop rows with null targets
df = df.dropna(subset=["val_accuracy"]).reset_index(drop=True)
print(f"After dropping NaN targets: {len(df):,} rows")
print(f"\nTarget (val_accuracy / memory_bytes) summary:")
print(df["val_accuracy"].describe().to_string())

## 2. Feature Engineering

Build two feature sets:
1. **TF-IDF**: Character n-grams from raw source code (captures syntax patterns like `append`, `malloc`, `new`)
2. **Structural**: Computed code metrics (LOC, nesting depth, vocabulary)

In [ ]:
print("Building TF-IDF feature matrix...")
print("  Using character n-grams (2,4) to capture syntax density")

tfidf_features = min(3000, len(df))  # Reduce for validation
X_tfidf, vectorizer = compute_tfidf_features(
    df['input'],
    max_features=tfidf_features,
    ngram_range=(2, 4),
    analyzer='char_wb',
)
print(f"  TF-IDF shape: {X_tfidf.shape}")

# Show top features
feature_names = vectorizer.get_feature_names_out()
print(f"  Sample features: {list(feature_names[:20])}")

In [ ]:
print("\nComputing structural code metrics...")
code_metrics = compute_code_metrics_batch(df['input'])
print(f"  Structural features: {list(code_metrics.columns)}")
print(code_metrics.describe().to_string())

In [ ]:
from scipy.sparse import hstack, csr_matrix

# Convert structural features to sparse
X_structural = csr_matrix(code_metrics.values)

# Combine TF-IDF + structural
X_combined = hstack([X_tfidf, X_structural])
all_feature_names = list(feature_names) + list(code_metrics.columns)

y = df['val_accuracy'].values

print(f"\nCombined feature matrix: {X_combined.shape}")
print(f"Target vector length: {len(y)}")
print(f"Feature count: TF-IDF={X_tfidf.shape[1]} + Structural={X_structural.shape[1]} = {X_combined.shape[1]}")

## 3. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]:,} samples")
print(f"Test set:     {X_test.shape[0]:,} samples")
print(f"\nTraining target stats:")
print(f"  Mean: {y_train.mean():.2f}")
print(f"  Std:  {y_train.std():.2f}")
print(f"  Range: [{y_train.min():.2f}, {y_train.max():.2f}]")

## 4. Train XGBoost Regressor

Using pseudo-Huber loss for gradient robustness against heavy-tailed memory targets.

In [ ]:
print("Training XGBoost regressor...")
print("  Objective: reg:pseudohubererror (robust to outliers)")

model = xgb.XGBRegressor(
    objective='reg:pseudohubererror',
    n_estimators=200 if not VALIDATION_MODE else 50,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=1,
    n_jobs=-1,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=10 if not VALIDATION_MODE else 5,
)

print("\n\u2705 Training complete!")

## 5. Evaluation: Spearman ρ and Kendall τ

In [ ]:
y_pred = model.predict(X_test)

# Spearman rank correlation
spearman_rho, spearman_p = stats.spearmanr(y_test, y_pred)

# Kendall tau
kendall_tau, kendall_p = stats.kendalltau(y_test, y_pred)

# Additional metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("="*70)
print("XGBoost REGRESSION RESULTS")
print("="*70)
print(f"\n  Spearman \u03c1:  {spearman_rho:.4f}  (p = {spearman_p:.2e})")
print(f"  Kendall \u03c4:   {kendall_tau:.4f}  (p = {kendall_p:.2e})")
print(f"  R\u00b2 Score:    {r2:.4f}")
print(f"  MAE:         {mae:.2f}")
print(f"  RMSE:        {rmse:.2f}")
print()

if spearman_rho >= 0.30:
    print("\u2705 Signal verification PASSED!")
    print(f"   Spearman \u03c1 = {spearman_rho:.4f} >= 0.30")
    print("   The dataset contains a valid, learnable signal linking")
    print("   lexical code syntax to physical performance metrics.")
else:
    print("\u26a0\ufe0f  Spearman \u03c1 < 0.30 \u2014 signal may be weak.")
    if VALIDATION_MODE:
        print("   This may be due to limited validation data. Try full mode.")

## 6. Visualization

In [ ]:
fig = plot_actual_vs_predicted(
    y_test, y_pred,
    title=f'XGBoost: Actual vs Predicted Memory (\u03c1={spearman_rho:.3f})'
)
plt.show()

In [ ]:
importances = model.feature_importances_

fig = plot_feature_importance(
    importances, all_feature_names,
    top_n=min(20, len(all_feature_names)),
    title='Top Features for Memory Prediction'
)
plt.show()

# Print top features
top_idx = np.argsort(importances)[-20:]
print("\nTop 20 most important features:")
for i in reversed(top_idx):
    print(f"  {all_feature_names[i]:30s} importance={importances[i]:.4f}")

## 7. Residual Analysis

In [ ]:
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residual distribution
axes[0].hist(residuals, bins=50, color=COLORS['highlight'], alpha=0.7, edgecolor='black')
axes[0].axvline(0, color=COLORS['primary'], linestyle='--', linewidth=2)
axes[0].set_title('Residual Distribution', fontweight='bold')
axes[0].set_xlabel('Residual (actual - predicted)')
axes[0].set_ylabel('Count')
axes[0].grid(True, alpha=0.2)

# Residuals vs predicted
if len(y_pred) > 3000:
    idx = np.random.choice(len(y_pred), 3000, replace=False)
else:
    idx = np.arange(len(y_pred))
axes[1].scatter(y_pred[idx], residuals[idx], alpha=0.3, s=10, color=COLORS['highlight'])
axes[1].axhline(0, color=COLORS['primary'], linestyle='--', linewidth=2)
axes[1].set_title('Residuals vs Predicted', fontweight='bold')
axes[1].set_xlabel('Predicted Memory (bytes)')
axes[1].set_ylabel('Residual')
axes[1].grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

print(f"\nResidual statistics:")
print(f"  Mean: {residuals.mean():.2f}")
print(f"  Std:  {residuals.std():.2f}")
print(f"  Median: {np.median(residuals):.2f}")

## Conclusion

In [ ]:
print("="*80)
print("EXPERIMENT 3 CONCLUSION")
print("="*80)
print()
print(f"XGBoost Results on {'validation' if VALIDATION_MODE else 'full'} dataset:")
print(f"  Spearman \u03c1 = {spearman_rho:.4f}")
print(f"  Kendall \u03c4  = {kendall_tau:.4f}")
print(f"  R\u00b2         = {r2:.4f}")
print()
print("This proves that a classical, non-neural model can extract a")
print("meaningful regression signal from source code syntax,")
print("validating the dataset's core efficacy independently of the")
print("RLM neural architecture.")
if VALIDATION_MODE:
    print("\n\u26a0\ufe0f  Note: Results from validation mode with limited data.")
    print("   Run with CODE_REGRESSION_FULL=1 for statistically powerful results.")